# EXP10: 周内择时 (Day of Week)

知识库标注 day_of_week IC=+0.033，WF=100%，最稳定因子之一。

1. 按周几分解 PnL/WR/Sharpe
2. 按年份 x 周几交叉稳定性
3. 跳过亏损日模拟
4. 按策略 x 周几分析

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)
trades = baseline['_trades']
print(f"Baseline: {len(trades)} trades, PnL=${sum(t.pnl for t in trades):.0f}")

In [ ]:
DOW_NAMES = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

df = pd.DataFrame([{
    'year': t.entry_time.year,
    'dow': t.entry_time.weekday(),
    'dow_name': DOW_NAMES.get(t.entry_time.weekday(), '?'),
    'pnl': t.pnl,
    'strategy': t.strategy,
    'direction': t.direction,
    'bars_held': t.bars_held,
    'win': 1 if t.pnl > 0 else 0,
} for t in trades])

## Part 1: 按周几 PnL 分解

In [ ]:
dow_stats = df.groupby('dow_name').agg(
    count=('pnl', 'count'),
    total_pnl=('pnl', 'sum'),
    avg_pnl=('pnl', 'mean'),
    win_rate=('win', 'mean'),
    avg_bars=('bars_held', 'mean'),
).round(2)
dow_stats = dow_stats.reindex(['Mon', 'Tue', 'Wed', 'Thu', 'Fri'])

print("=== PnL by Day of Week ===")
print(dow_stats)
print(f"\nBest day: {dow_stats['avg_pnl'].idxmax()} (${dow_stats['avg_pnl'].max():.2f}/trade)")
print(f"Worst day: {dow_stats['avg_pnl'].idxmin()} (${dow_stats['avg_pnl'].min():.2f}/trade)")

## Part 2: 年份 x 周几 交叉稳定性

In [ ]:
pivot_pnl = df.pivot_table(index='year', columns='dow_name', values='pnl', aggfunc='sum')
pivot_pnl = pivot_pnl[['Mon', 'Tue', 'Wed', 'Thu', 'Fri']]
print("=== Total PnL by Year x Day ===")
print(pivot_pnl.round(0))

print("\n=== Avg PnL/trade by Year x Day ===")
pivot_avg = df.pivot_table(index='year', columns='dow_name', values='pnl', aggfunc='mean')
pivot_avg = pivot_avg[['Mon', 'Tue', 'Wed', 'Thu', 'Fri']]
print(pivot_avg.round(2))

print("\n=== Day Consistency (years with positive PnL) ===")
for day in ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']:
    col = pivot_pnl[day].dropna()
    pos = (col > 0).sum()
    print(f"  {day}: {pos}/{len(col)} years positive, total ${col.sum():.0f}")

## Part 3: 跳过亏损日模拟

In [ ]:
def filter_by_days(trades, allowed_days, label):
    filtered = [t for t in trades if t.entry_time.weekday() in allowed_days]
    total_pnl = sum(t.pnl for t in filtered)
    n = len(filtered)
    wins = sum(1 for t in filtered if t.pnl > 0)
    wr = wins / n * 100 if n > 0 else 0
    avg = total_pnl / n if n > 0 else 0
    print(f"  {label:30s}: N={n:5d}  PnL=${total_pnl:8.0f}  $/trade={avg:6.2f}  WR={wr:5.1f}%")

print("=== Day Filter Scenarios ===")
scenarios = [
    ([0,1,2,3,4], "All days (baseline)"),
    ([1,2,3,4], "Skip Monday"),
    ([0,1,2,3], "Skip Friday"),
    ([0,2,3,4], "Skip Tuesday"),
    ([0,1,3,4], "Skip Wednesday"),
    ([0,1,2,4], "Skip Thursday"),
    ([1,2,3], "Tue-Thu only"),
    ([0,1,2], "Mon-Wed only"),
    ([2,3,4], "Wed-Fri only"),
]
for days, label in scenarios:
    filter_by_days(trades, days, label)

print("\n*** 注意: 后处理过滤是上限估计 ***")

## Part 4: 策略 x 周几

In [ ]:
print("=== PnL by Strategy x Day ===")
pivot_strat = df.pivot_table(index='strategy', columns='dow_name', values='pnl', aggfunc='sum')
pivot_strat = pivot_strat[['Mon', 'Tue', 'Wed', 'Thu', 'Fri']]
print(pivot_strat.round(0))

print("\n=== Direction x Day ===")
pivot_dir = df.pivot_table(index='direction', columns='dow_name', values='pnl', aggfunc='sum')
pivot_dir = pivot_dir[['Mon', 'Tue', 'Wed', 'Thu', 'Fri']]
print(pivot_dir.round(0))

print("\n=== Trade Count by Strategy x Day ===")
pivot_n = df.pivot_table(index='strategy', columns='dow_name', values='pnl', aggfunc='count')
pivot_n = pivot_n[['Mon', 'Tue', 'Wed', 'Thu', 'Fri']]
print(pivot_n)